# Загрузка котировок с MOEX ISS

Дневные данные по 25 ликвидным акциям (режим торгов TQBR)
с 2014 года. API отдаёт максимум 100 строк за запрос, поэтому
загрузчик обходит пагинацию циклом. Функция вынесена в `src/data.py`.

Результат сохраняется в `data/raw/moex_prices.csv`.

In [40]:
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import TICKERS, load_history, project_root

RAW = project_root() / "data" / "raw"

In [23]:
sber = load_history("SBER")

print(sber.shape)
print(sber["TRADEDATE"].min(), sber["TRADEDATE"].max())
print("пропусков в CLOSE:", sber["CLOSE"].isna().sum())

sber[["TRADEDATE", "SECID", "CLOSE", "VOLUME"]].head()

(3190, 24)
2014-01-06 00:00:00 2026-08-19 00:00:00
пропусков в CLOSE: 18


,TRADEDATE,SECID,CLOSE,VOLUME
0,2014-01-06,SBER,98.91,31691800
1,2014-01-08,SBER,98.19,42372290
2,2014-01-09,SBER,98.00,45986900
3,2014-01-10,SBER,99.20,51902400
4,2014-01-13,SBER,100.25,62051250


### Пропуски в данных

18 дней без цены закрытия у SBER: остановка торгов на МосБирже
(конец февраля — март 2022) и отдельные праздничные дни.
Во всех случаях VOLUME = 0 и NUMTRADES = 0 — торгов не было.

Решение: строки без CLOSE удаляем. Заполнять их последней
известной ценой нельзя — это создало бы несуществующие
наблюдения и исказило распределение доходностей по дням недели.

In [24]:
missing = sber[sber["CLOSE"].isna()]
missing[["TRADEDATE", "CLOSE", "VOLUME", "NUMTRADES"]]

,TRADEDATE,CLOSE,VOLUME,NUMTRADES
2019,2022-01-07,NaN,0,0
2052,2022-02-23,NaN,0,0
2055,2022-02-28,NaN,0,0
2056,2022-03-01,NaN,0,0
2057,2022-03-02,NaN,0,0
2058,2022-03-03,NaN,0,0
2059,2022-03-04,NaN,0,0
2060,2022-03-09,NaN,0,0
2061,2022-03-10,NaN,0,0
2062,2022-03-11,NaN,0,0


In [34]:
frames = []

for i, ticker in enumerate(TICKERS, 1):
    df = load_history(ticker)
    frames.append(df)
    print(f"{i}/{len(TICKERS)} {ticker}: {len(df)} строк")
    time.sleep(0.5)

prices = pd.concat(frames, ignore_index=True)
print(prices.shape)

1/25 SBER: 3190 строк
2/25 GAZP: 3084 строк
3/25 LKOH: 3190 строк
4/25 GMKN: 3084 строк
5/25 NVTK: 3190 строк
6/25 ROSN: 3084 строк
7/25 TATN: 3190 строк
8/25 PLZL: 3084 строк
9/25 SNGSP: 3084 строк
10/25 X5: 410 строк
11/25 MGNT: 3190 строк
12/25 CHMF: 3084 строк
13/25 NLMK: 3084 строк
14/25 ALRS: 3190 строк
15/25 AFLT: 3190 строк
16/25 IRAO: 3180 строк
17/25 RTKM: 3190 строк
18/25 MOEX: 3190 строк
19/25 PHOR: 3190 строк
20/25 VTBR: 3190 строк
21/25 SIBN: 3084 строк
22/25 SMLT: 1474 строк
23/25 POSI: 1185 строк
24/25 MAGN: 3084 строк
25/25 T: 437 строк
(69532, 24)


In [37]:
summary = prices.groupby("SECID")["TRADEDATE"].agg(["min", "max", "count"])
summary.sort_values("count")

,min,max,count
SECID,,,
X5,2025-01-09,2026-08-19,410
T,2024-11-28,2026-08-19,437
POSI,2021-12-17,2026-08-19,1185
SMLT,2020-10-29,2026-08-19,1474
CHMF,2014-06-09,2026-08-19,3084
GAZP,2014-06-09,2026-08-19,3084
GMKN,2014-06-09,2026-08-19,3084
MAGN,2014-06-09,2026-08-19,3084
SNGSP,2014-06-09,2026-08-19,3084


In [38]:
RAW.mkdir(parents=True, exist_ok=True)

path = RAW / "moex_prices.csv"
prices.to_csv(path, index=False)

#print(path)

print(f"{path.stat().st_size / 1024**2:.1f} МБ")

11.4 МБ


### Состав выборки

- Все 25 бумаг торгуются по последний день выборки, делистинга нет.
- Четыре бумаги с короткой историей: X5 (407 дней), T (434),
  POSI (1182), SMLT (1471) — недавние IPO и смены тикера.
- Девять бумаг начинаются с 2014-06-09: перевод на режим Т+2,
  до этой даты они торговались в другом режиме торгов,
  которого нет в нашей выгрузке.

Решение: оставляем все бумаги. Единица наблюдения —
«бумага в конкретный день», разная длина истории её не искажает.
Следствие: поздние годы представлены большим числом бумаг,
поэтому устойчивость результата проверяем отдельно по подпериодам.